In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

from analysis.gravnet.model import NeutrinoGravNetRegressionFASER
from analysis.utils.utils import get_torch_path, get_weights_path, get_figures_path

# Must be a truth3 or truth4 run
WEIGHTS_DIR = "gravnet_regression_faser_all_events_mean_reparam_std_truth3"
RUN = 10000
DATA_TYPE = "all"

assert "_truth3" in WEIGHTS_DIR or "_truth4" in WEIGHTS_DIR, \
    "This notebook is for truth-label runs only. Set WEIGHTS_DIR to a truth3 or truth4 run."

TRUTH_CLASSES = 3 if "_truth3" in WEIGHTS_DIR else 4
# truth3 labels: 0=hadronic/other, 1=electron (merged), 2=muon
# truth4 labels: 0=hadronic/other, 1=secondary_e (delta-ray), 2=primary_EM_e, 3=muon
CLASS_NAMES = (
    ["hadronic", "electron", "muon"] if TRUTH_CLASSES == 3
    else ["hadronic", "secondary_e", "primary_EM_e", "muon"]
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Truth classes: {TRUTH_CLASSES} — {CLASS_NAMES}")

weights_path = get_weights_path() / WEIGHTS_DIR
torch_path   = get_torch_path()
figures_path = get_figures_path() / WEIGHTS_DIR
figures_path.mkdir(parents=True, exist_ok=True)

print(f"Weights : {weights_path}")
print(f"Figures : {figures_path}")

**Load model and val dataset**

In [ ]:
checkpoint = torch.load(weights_path / "best_model.pt", weights_only=False)
print(f"Best model from epoch {checkpoint['epoch'] + 1},  "
      f"val_loss={checkpoint['val_loss']:.4f}")

pooling   = "sum" if "_sum_" in WEIGHTS_DIR else "mean"
input_dim = 1 + TRUTH_CLASSES  # log10(n_hits) + one-hot truth
print(f"Pooling: {pooling},  input_dim: {input_dim}")

model = NeutrinoGravNetRegressionFASER(
    input_dim=input_dim,
    num_targets=2,
    faser_dim=5,
    pooling=pooling,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
norm_stats = checkpoint.get("norm_stats", None)

# Load val set (same split as training)
run_str  = "nue"
run_path = torch_path / f"{RUN}/pointnetpp_faser_{DATA_TYPE}_events"
dataset  = []
for f in sorted(run_path.glob(f"{run_str}_*.pt")):
    dataset.extend(torch.load(f, weights_only=False))
print(f"Loaded {len(dataset)} events.")

_, val_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
print(f"Val set: {len(val_dataset)} events.")

# Augment with truth labels (same as training)
label_attr = "y" if TRUTH_CLASSES == 3 else "pdg_label"
for data in val_dataset:
    y_oh = F.one_hot(getattr(data, label_attr), num_classes=TRUTH_CLASSES).float()
    data.x = torch.cat([data.x, y_oh], dim=1)

print(f"Node feature shape after augmentation: {val_dataset[0].x.shape}  (should be [N, {input_dim}])")

**Input gradient saliency**

For each event, compute `|∂output/∂x|` for every node and input channel.
Average across nodes and events gives a per-channel importance score.

Channel layout: `[0: log10(n_hits), 1...: one-hot truth label]`

If truth channels have large gradients → model relies on them heavily.
If near zero → model ignores them (performance gain from truth labels must come from some other mechanism, or isn't real).

In [ ]:
import seaborn as sns
sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)

N_EVENTS = min(500, len(val_dataset))  # subsample for speed

# Accumulate |grad| per channel
grad_sum   = np.zeros(input_dim)   # sum of mean |grad| per event
grad_sum_sq = np.zeros(input_dim)  # for std
n_events   = 0

model.eval()
for data in val_dataset[:N_EVENTS]:
    data = data.to(device)
    x    = data.x.clone().requires_grad_(True)
    batch = torch.zeros(x.shape[0], dtype=torch.long, device=device)

    out = model(x, data.pos, batch, data.x_faser.unsqueeze(0))
    # Gradient w.r.t. each output (log10 E_nu and logit y) summed
    loss = out.sum()
    loss.backward()

    with torch.no_grad():
        # Mean |grad| over nodes for this event, shape [input_dim]
        mean_grad = x.grad.abs().mean(dim=0).cpu().numpy()
    grad_sum    += mean_grad
    grad_sum_sq += mean_grad ** 2
    n_events    += 1

grad_mean = grad_sum / n_events
grad_std  = np.sqrt(np.maximum(grad_sum_sq / n_events - grad_mean ** 2, 0))

channel_names = ["log10(n_hits)"] + [f"label={c}\n({CLASS_NAMES[c]})" for c in range(TRUTH_CLASSES)]

fig, ax = plt.subplots(figsize=(7, 4))
x_pos = np.arange(input_dim)
colors = ["#353D4C"] + ["#E17883", "#5691D9", "#a5deb6", "#c9a0dc"][:TRUTH_CLASSES]
ax.bar(x_pos, grad_mean, yerr=grad_std, color=colors, alpha=0.85,
       capsize=4, edgecolor='white', linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(channel_names, fontsize=9)
ax.set_ylabel(r"Mean $|\partial\, \mathrm{output} / \partial x_i|$")
ax.set_title(f"Input gradient saliency  (N={n_events} events)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis="y", linestyle=":", linewidth=0.8, alpha=0.4)
plt.tight_layout()
plt.savefig(figures_path / "saliency_input_grad.png", dpi=350, bbox_inches="tight")
plt.show()
print("Saved: saliency_input_grad.png")
print()
for name, mean, std in zip(channel_names, grad_mean, grad_std):
    short = name.replace("\n", " ")
    print(f"  {short:<35s}  {mean:.4f} ± {std:.4f}")

**Ablation: zero out truth label channels at inference**

After training, replace the truth label one-hot columns with zeros and re-run inference.
The performance drop (relative to full truth labels) reveals how much the model actually
exploits the truth information vs. the hit count channel alone.

In [ ]:
def preds_to_physical(preds_raw, norm_stats):
    if norm_stats is not None:
        log_E_nu = preds_raw[:, 0] * norm_stats["sigma_enu"]   + norm_stats["mu_enu"]
        logit_y  = preds_raw[:, 1] * norm_stats["sigma_logit"] + norm_stats["mu_logit"]
    else:
        log_E_nu = preds_raw[:, 0]
        logit_y  = preds_raw[:, 1]
    E_nu  = 10 ** log_E_nu
    y     = torch.sigmoid(logit_y)
    E_lep = y * E_nu
    E_roe = (1 - y) * E_nu
    return torch.stack([E_nu, E_lep, E_roe], dim=1).numpy()


def run_inference(loader, zero_truth_channels=False):
    all_preds, all_targets = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            x = data.x.clone()
            if zero_truth_channels:
                x[:, 1:] = 0.0  # zero out all truth one-hot columns
            raw     = model(x, data.pos, data.batch, data.x_faser)
            targets = torch.stack([data.E_nu, data.E_lepton, data.E_roe], dim=1)
            all_preds.append(raw.cpu())
            all_targets.append(targets.cpu())
    preds_raw      = torch.cat(all_preds)
    targets_linear = torch.cat(all_targets).numpy()
    preds_linear   = preds_to_physical(preds_raw, norm_stats)
    return preds_linear, targets_linear


val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

preds_full,  targets = run_inference(val_loader, zero_truth_channels=False)
preds_zeroed, _      = run_inference(val_loader, zero_truth_channels=True)

TARGET_NAMES  = ["E_nu", "E_lepton", "E_roe"]
TARGET_LATEX  = [r"$E_\nu$", r"$E_\mathrm{lep}$", r"$E_\mathrm{roe}$"]

print(f"{'Target':<12} {'Full truth σ':>13} {'Zeroed σ':>10} {'Δσ':>8}")
for i, name in enumerate(TARGET_NAMES):
    yt  = targets[:, i]
    res_full   = (preds_full[:, i]   - yt) / yt
    res_zeroed = (preds_zeroed[:, i] - yt) / yt
    s_full   = np.std(res_full)
    s_zeroed = np.std(res_zeroed)
    print(f"{name:<12} {s_full:>13.3f} {s_zeroed:>10.3f} {s_zeroed - s_full:>+8.3f}")

# Plot residual distributions: full vs zeroed
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
COLORS = ["#353D4C", "#E17883", "#5691D9"]
for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax = axes[i]
    yt = targets[:, i]
    res_f = (preds_full[:, i]   - yt) / yt
    res_z = (preds_zeroed[:, i] - yt) / yt
    xlim  = min(np.percentile(np.abs(np.concatenate([res_f, res_z])), 99), 5.0)
    bins  = np.linspace(-xlim, xlim, 60)
    ax.hist(res_f, bins=bins, histtype="stepfilled", alpha=0.5, color=COLORS[i],
            density=True, label=f"Full truth  σ={np.std(res_f):.3f}")
    ax.hist(res_z, bins=bins, histtype="step", linewidth=1.5, color="gray",
            density=True, label=f"Truth zeroed σ={np.std(res_z):.3f}")
    ax.set_xlabel(rf"({latex}$^{{\rm reco}}$ − true) / true")
    ax.set_ylabel("Density")
    ax.set_title(name)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.3)
plt.suptitle("Ablation: full truth labels vs zeroed truth channels", fontsize=12)
plt.tight_layout()
plt.savefig(figures_path / "ablation_truth_channels.png", dpi=350, bbox_inches="tight")
plt.show()
print("Saved: ablation_truth_channels.png")

**Resolution breakdown by dominant node class**

For each event, find the majority truth label among its nodes (the class with the most nodes).
Split events by majority class and compare reconstruction quality.

For νe: most events will be majority-electron. Rare hadronic-dominated events
are likely NC contamination or unusual topology.

In [ ]:
# Compute per-event majority class and class fractions
# Note: data.x columns are [log10(n_hits), one_hot_0, ..., one_hot_{C-1}]
# The original label_attr is stored in data.y (truth3) or data.pdg_label (truth4)
rows = []
for i, data in enumerate(val_dataset):
    labels = getattr(data, label_attr)  # [N] node labels (un-augmented)
    # label_attr was set earlier; labels are still accessible on the Data object
    counts = torch.bincount(labels, minlength=TRUTH_CLASSES).float()
    fracs  = counts / counts.sum()
    majority = int(counts.argmax())
    row = {
        "majority_class": majority,
        "E_nu":  float(data.E_nu),
        "E_lep": float(data.E_lepton),
        "E_roe": float(data.E_roe),
    }
    for c in range(TRUTH_CLASSES):
        row[f"frac_{CLASS_NAMES[c]}"] = float(fracs[c])
    for j, name in enumerate(TARGET_NAMES):
        yt = float([data.E_nu, data.E_lepton, data.E_roe][j])
        yp = float(preds_full[i, j])
        row[f"res_{name}"] = (yp - yt) / max(yt, 1e-6)
    rows.append(row)

import pandas as pd
df = pd.DataFrame(rows)

print(f"Events per majority class:")
for c, name in enumerate(CLASS_NAMES):
    n = (df["majority_class"] == c).sum()
    print(f"  {name:<20s}: {n} ({100*n/len(df):.1f}%)")

# Residual std per majority class
fig, axes = plt.subplots(1, len(TARGET_NAMES), figsize=(15, 4))
CLASS_COLORS = ["#b0b7bc", "#e05c5c", "#5b8db8", "#c9a0dc"][:TRUTH_CLASSES]
for j, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax = axes[j]
    col = f"res_{name}"
    all_vals = df[col].values
    xlim = min(np.percentile(np.abs(all_vals), 99), 5.0)
    bins = np.linspace(-xlim, xlim, 50)
    for c, (cname, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
        mask = df["majority_class"] == c
        if mask.sum() < 10:
            continue
        vals = df.loc[mask, col].values
        label = f"{cname} (N={mask.sum()}, σ={np.std(vals):.3f})"
        ax.hist(vals, bins=bins, histtype="stepfilled", alpha=0.4, color=color, density=True, label=label)
        ax.hist(vals, bins=bins, histtype="step", linewidth=1.2, color=color, density=True)
    ax.set_xlabel(rf"({latex}$^{{\rm reco}}$ − true) / true")
    ax.set_ylabel("Density")
    ax.set_title(name)
    ax.legend(fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.3)
plt.suptitle("Resolution by majority node class", fontsize=12)
plt.tight_layout()
plt.savefig(figures_path / "resolution_by_majority_class.png", dpi=350, bbox_inches="tight")
plt.show()
print("Saved: resolution_by_majority_class.png")

**Resolution vs class fraction**

Scatter: fraction of EM shower nodes vs |rel error| on E_lepton.
If primary EM shower fraction correlates with better E_lepton reconstruction,
that supports the value of the truth4 (4-class) distinction.

In [ ]:
# Plot |res_E_lep| vs fraction of each class
df["abserr_Elep"] = df["res_E_lep"].abs() if "res_E_lep" in df.columns else df["res_E_lepton"].abs()
df["abserr_Elep"] = (preds_full[:, 1] - np.array([r["E_lep"] for r in rows])) / np.array([max(r["E_lep"], 1e-6) for r in rows])
df["abserr_Elep"] = np.abs(df["abserr_Elep"])

frac_cols  = [f"frac_{c}" for c in CLASS_NAMES]
frac_latex = [f"Fraction {c} nodes" for c in CLASS_NAMES]

fig, axes = plt.subplots(1, TRUTH_CLASSES, figsize=(5 * TRUTH_CLASSES, 4))
if TRUTH_CLASSES == 1:
    axes = [axes]
for ax, col, label, color in zip(axes, frac_cols, frac_latex, CLASS_COLORS):
    r = np.corrcoef(df[col].values, df["abserr_Elep"].values)[0, 1]
    ax.scatter(df[col].values, df["abserr_Elep"].values,
               s=4, alpha=0.3, color=color, linewidths=0)
    ax.set_xlabel(label)
    ax.set_ylabel(r"|rel error| $E_\mathrm{lep}$")
    ax.set_ylim(0, min(df["abserr_Elep"].quantile(0.99) * 1.1, 5.0))
    ax.set_title(f"Pearson r = {r:.3f}")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.3)
plt.suptitle(r"$|$rel error$|$ on $E_\mathrm{lep}$ vs node class fraction", fontsize=12)
plt.tight_layout()
plt.savefig(figures_path / "elep_err_vs_class_fraction.png", dpi=350, bbox_inches="tight")
plt.show()
print("Saved: elep_err_vs_class_fraction.png")